# bench_fin — Next-day return classification with augmentations + SSL

This notebook trains S&P 500 next-day return classifiers using the shared
`BenchmarkRunner`, augmentation utilities, and semi-supervised wrappers. Training
uses Andrew MVD's S&P 500 dataset, while evaluation on the World Stock Prices
feed measures transfer.


## 1. Environment


In [ ]:
%%capture
!pip install -q kaggle xgboost lightgbm imbalanced-learn

## 2. Imports and seeds


In [ ]:
from collections import Counter
from itertools import cycle
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.models import MODEL_REGISTRY
from pipelines_torch.ss_models import SemiSupervisedTabular
from data_augmentation.augmentations import MGS_GRF_Augmentor
from pipelines_torch.base import SimplePredictor
from utils.metrics import accuracy_score, f1_score, precision_score, recall_score
from utils.utils import load_model
from ml_pipeline.data_augmentation.augmentations import smote_augmentation, mixup_smote_augmentation

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 3. Download datasets


In [ ]:
from pathlib import Path

from utils.kaggle_utils import ensure_kaggle_dataset

DATA_ROOT = Path("data_fin")
SP500_DATASET = "andrewmvd/sp-500-stocks"
WORLD_DATASET = "nelgiriyewithana/world-stock-prices-daily-updating"

SP500_DIR = ensure_kaggle_dataset(
    dataset_slug=SP500_DATASET,
    local_dir=DATA_ROOT / "sp500",
    description="S&P 500 stocks dataset",
    kaggle_subdir="sp-500-stocks",
)
WORLD_DIR = ensure_kaggle_dataset(
    dataset_slug=WORLD_DATASET,
    local_dir=DATA_ROOT / "world",
    description="World stock prices dataset",
    kaggle_subdir="world-stock-prices-daily-updating",
)

## 4. Feature engineering
We compute standard technical indicators and build binary/ternary labels for
next-day returns. The helper works for both datasets so we can share the same
feature space.


In [ ]:
def load_first_csv(directory: Path) -> pd.DataFrame:
    for csv_path in sorted(directory.glob("*.csv")):
        df = pd.read_csv(csv_path)
        if {'Date', 'Close'}.issubset(df.columns):
            return df
    raise FileNotFoundError(f"No CSV with Date/Close columns found in {directory}")


def build_features(df: pd.DataFrame, *, symbol_col: str | None = None, sector_col: str | None = None) -> pd.DataFrame:
    data = df.copy()
    data['Date'] = pd.to_datetime(data['Date'])
    if symbol_col is None:
        for candidate in ['Symbol', 'symbol', 'Ticker', 'Name']:
            if candidate in data.columns:
                symbol_col = candidate
                break
    if symbol_col is None:
        symbol_col = 'SYMBOL'
        data[symbol_col] = '_ONE_'
    if sector_col and sector_col not in data.columns:
        sector_col = None
    data = data.sort_values([symbol_col, 'Date'])
    data['ret'] = data.groupby(symbol_col)['Close'].pct_change()
    data['ret_next'] = data.groupby(symbol_col)['Close'].pct_change().shift(-1)
    data['hlv'] = (data['High'] - data['Low']) / data['Close']
    for window in (3, 5, 10):
        data[f'ma{window}'] = data.groupby(symbol_col)['Close'].transform(lambda s: s.rolling(window).mean())
        data[f'vol{window}'] = data.groupby(symbol_col)['ret'].transform(lambda s: s.rolling(window).std())
    data = data.dropna().copy()
    data['y_bin'] = (data['ret_next'] > 0).astype(int)
    quantiles = data['ret_next'].quantile([0.33, 0.66]).values
    data['y_3'] = np.select(
        [data['ret_next'] < quantiles[0], data['ret_next'] > quantiles[1]],
        [-1, 1],
        default=0,
    )
    data['sector'] = data[sector_col] if sector_col else '_UNK_'
    return data[[symbol_col, 'Date', 'sector', 'ret', 'hlv', 'ma3', 'ma5', 'ma10', 'vol3', 'vol5', 'vol10', 'y_bin', 'y_3']]

sp500_df = load_first_csv(SP500_DIR)
world_df = load_first_csv(WORLD_DIR)
sp500_features = build_features(sp500_df, sector_col='Sector' if 'Sector' in sp500_df.columns else None)
world_features = build_features(world_df)
sp500_features.head()


## 5. Supervised benchmark
We focus on the binary target for `BenchmarkRunner`. Augmentations are pulled
from the shared tabular utilities: standard SMOTE, Mixup-SMOTE, and the official
MGS-GRF wrapper.


In [ ]:
feature_cols = ['ret', 'hlv', 'ma3', 'ma5', 'ma10', 'vol3', 'vol5', 'vol10']
X = sp500_features[feature_cols].to_numpy(dtype=np.float32)
y = sp500_features['y_bin'].to_numpy(dtype=np.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)

minority_class = int(np.argmin(np.bincount(y_train)))


def smote_adapter(features: np.ndarray, labels: np.ndarray, *, max_factor: float = 2.0, random_state: int = SEED):
    return smote_augmentation(features, labels, random_state=random_state, max_factor=max_factor)


def mixup_smote_adapter(features: np.ndarray, labels: np.ndarray, *, max_factor: float = 2.0, random_state: int = SEED):
    return mixup_smote_augmentation(features, labels, n_samples=features.shape[0], alpha=0.2, random_state=random_state, max_factor=max_factor)


def mgs_grf_adapter(features: np.ndarray, labels: np.ndarray, *, max_factor: float = 2.0, random_state: int = SEED):
    counts = Counter(labels)
    majority = max(counts.values())
    target = int(majority / max_factor)
    minority_count = counts.get(minority_class, 0)
    n_samples = max(0, target - minority_count)
    if n_samples <= 0:
        return features, labels
    augmentor = MGS_GRF_Augmentor()
    X_aug, y_aug = augmentor.fit_resample(features, labels, target_class=minority_class, n_samples=n_samples)
    return X_aug, y_aug

augmentations = [None, smote_adapter, mixup_smote_adapter, mgs_grf_adapter]
metrics = [accuracy_score, f1_score, precision_score, recall_score]

model_configs = [
    {
        "name": "mlp_classifier",
        "class": MODEL_REGISTRY["mlp_classifier"],
        "params": {"input_dim": X_train.shape[1], "num_classes": 2},
    },
    {
        "name": "xgboost_classifier",
        "class": MODEL_REGISTRY["xgboost_classifier"],
        "params": {"n_estimators": 800, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "random_state": SEED},
    },
]

runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=augmentations,
    metrics=metrics,
    task_type="classification",
    device=DEVICE,
    epochs=8,
    batch_size=128,
    use_kfold=False,
    learning_rate=1e-3,
    path_start="bench_fin_supervised",
    random_state=SEED,
)
benchmark_results = runner.run(X_train, y_train)
benchmark_results


### Validate checkpoints on held-out S&P 500 samples


In [ ]:
def roc_auc_metric(y_true: np.ndarray, probs: np.ndarray) -> float:
    if probs.ndim == 1:
        return roc_auc_score(y_true, probs)
    if probs.shape[1] == 1:
        return roc_auc_score(y_true, probs[:, 0])
    return roc_auc_score(y_true, probs[:, 1])


def evaluate_binary_models(model_names: Iterable[str], aug_names: Iterable[str], X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    records = []
    for name in model_names:
        cfg = next(cfg for cfg in model_configs if cfg['name'] == name)
        for aug in aug_names:
            try:
                model = load_model(
                    cfg['class'],
                    cfg['name'],
                    cfg['params'],
                    path_start="bench_fin_supervised",
                    augmentation='none' if aug is None else aug.__name__,
                )
            except FileNotFoundError:
                continue
            predictor = SimplePredictor(model, task_type="classification", device=DEVICE, batch_size=256)
            probs = predictor.predict_proba(X)
            records.append({
                "model": name,
                "augmentation": 'none' if aug is None else aug.__name__,
                "accuracy": float(accuracy_score(y, probs)),
                "f1_macro": float(f1_score(y, probs)),
                "precision_macro": float(precision_score(y, probs)),
                "recall_macro": float(recall_score(y, probs)),
                "roc_auc": float(roc_auc_metric(y, probs)),
            })
    return pd.DataFrame.from_records(records)

val_metrics = evaluate_binary_models([cfg['name'] for cfg in model_configs], augmentations, X_val, y_val)
val_metrics.sort_values(["f1_macro", "accuracy"], ascending=False)


## 6. Transfer evaluation on World Stock Prices
We engineer the same features, apply the S&P scaler, and score the saved models.


In [ ]:
world_X = scaler.transform(world_features[feature_cols].to_numpy(dtype=np.float32))
world_y = world_features['y_bin'].to_numpy(dtype=np.int64)
world_metrics = evaluate_binary_models([cfg['name'] for cfg in model_configs], augmentations, world_X, world_y)
world_metrics.sort_values(["f1_macro", "accuracy"], ascending=False)


## 7. Semi-supervised Mean Teacher on tabular features
We mask 60% of the labels and call `SemiSupervisedTabular` (Mean Teacher mode)
to exploit unlabeled days.


In [ ]:
mask = np.random.default_rng(SEED).random(len(y_train)) < 0.6
X_labeled = X_train[~mask]
y_labeled = y_train[~mask]
X_unlabeled = X_train[mask]

labeled_dataset = TensorDataset(torch.tensor(X_labeled), torch.tensor(y_labeled))
unlabeled_dataset = TensorDataset(torch.tensor(X_unlabeled))
val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

labeled_loader = DataLoader(labeled_dataset, batch_size=128, shuffle=True)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=256, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)


In [ ]:
def train_fin_ssl(epochs: int = 15, lr: float = 3e-4):
    base_model = MODEL_REGISTRY["mlp_classifier"](input_dim=X_train.shape[1], num_classes=2)
    ssl_model = SemiSupervisedTabular(base_model, num_classes=2, use_mean_teacher=True).to(DEVICE)
    optimizer = torch.optim.AdamW(ssl_model.parameters(), lr=lr, weight_decay=1e-4)
    history = []
    if len(unlabeled_loader) == 0:
        raise ValueError("No unlabeled samples available for SSL training")
    unlabeled_iter = cycle(unlabeled_loader)
    for epoch in range(epochs):
        ssl_model.train()
        for xb_l, yb_l in labeled_loader:
            xb_u = next(unlabeled_iter)[0]
            xb_l = xb_l.to(DEVICE)
            yb_l = yb_l.to(DEVICE)
            xb_u = xb_u.to(DEVICE)
            optimizer.zero_grad()
            loss, _ = ssl_model.step((xb_l, yb_l), (xb_u, None), epoch)
            loss.backward()
            optimizer.step()
            ssl_model.post_step()
        ssl_model.eval()
        with torch.no_grad():
            logits = ssl_model(torch.tensor(X_val, dtype=torch.float32, device=DEVICE))
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            metrics = {
                "accuracy": float(accuracy_score(y_val, preds)),
                "f1_macro": float(f1_score(y_val, probs)),
            }
        history.append(metrics)
        print(f"Epoch {epoch+1:02d}: val F1={metrics['f1_macro']:.3f}")
    return ssl_model, pd.DataFrame(history)


In [ ]:
ssl_model, ssl_history = train_fin_ssl(epochs=15, lr=3e-4)
ssl_history


In [ ]:
def evaluate_ssl_model(model: torch.nn.Module, X: np.ndarray, y: np.ndarray) -> pd.Series:
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32, device=DEVICE))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = probs.argmax(axis=1)
    return pd.Series({
        "accuracy": accuracy_score(y, preds),
        "f1_macro": f1_score(y, probs),
        "roc_auc": roc_auc_metric(y, probs),
    })

ssl_val_metrics = evaluate_ssl_model(ssl_model, X_val, y_val)
ssl_world_metrics = evaluate_ssl_model(ssl_model, world_X, world_y)
ssl_val_metrics, ssl_world_metrics


### Multi-class extension
To evaluate the three-class problem, swap `y_bin` for `y_3` in the cells above and
adjust the metrics (macro F1 works out-of-the-box). The augmentation adapters remain
valid because they derive the target class per run.
